In [9]:
import torch
import torch.nn as nn
import tad_dftd3 as d3
import tad_mctc as mctc

sample1 = dict(
    numbers=mctc.convert.symbol_to_number("Pb H H H H Bi H H H".split()),
    positions=torch.tensor(
        [
            [-0.00000020988889, -4.98043478877778, +0.00000000000000],
            [+3.06964045311111, -6.06324400177778, +0.00000000000000],
            [-1.53482054188889, -6.06324400177778, -2.65838526500000],
            [-1.53482054188889, -6.06324400177778, +2.65838526500000],
            [-0.00000020988889, -1.72196703577778, +0.00000000000000],
            [-0.00000020988889, +4.77334244722222, +0.00000000000000],
            [+1.35700257511111, +6.70626379422222, -2.35039772300000],
            [-2.71400388988889, +6.70626379422222, +0.00000000000000],
            [+1.35700257511111, +6.70626379422222, +2.35039772300000],
        ],
        dtype=torch.float64,
        device=torch.device("cpu"),
    ),
)
sample2 = dict(
    numbers=mctc.convert.symbol_to_number(
        "C C C C C C I H H H H H S H C H H H".split(" ")
    ),
    positions=torch.tensor(
        [
            [-1.42754169820131, -1.50508961850828, -1.93430551124333],
            [+1.19860572924150, -1.66299114873979, -2.03189643761298],
            [+2.65876001301880, +0.37736955363609, -1.23426391650599],
            [+1.50963368042358, +2.57230374419743, -0.34128058818180],
            [-1.12092277855371, +2.71045691257517, -0.25246348639234],
            [-2.60071517756218, +0.67879949508239, -1.04550707592673],
            [-2.86169588073340, +5.99660765711210, +1.08394899986031],
            [+2.09930989272956, -3.36144811062374, -2.72237695164263],
            [+2.64405246349916, +4.15317840474646, +0.27856972788526],
            [+4.69864865613751, +0.26922271535391, -1.30274048619151],
            [-4.63786461351839, +0.79856258572808, -0.96906659938432],
            [-2.57447518692275, -3.08132039046931, -2.54875517521577],
            [-5.88211879210329, 11.88491819358157, +2.31866455902233],
            [-8.18022701418703, 10.95619984550779, +1.83940856333092],
            [-5.08172874482867, 12.66714386256482, -0.92419491629867],
            [-3.18311711399702, 13.44626574330220, -0.86977613647871],
            [-5.07177399637298, 10.99164969235585, -2.10739192258756],
            [-6.35955320518616, 14.08073002965080, -1.68204314084441],
        ],
        dtype=torch.float64,
        device=torch.device("cpu"),
    ),
)
numbers = mctc.batch.pack(
    (
        sample1["numbers"],
        sample2["numbers"],
    )
)
positions = mctc.batch.pack(
    (
        sample1["positions"],
        sample2["positions"],
    )
)


class Model(nn.Module):
    """
    Fully connected neural network (dense network)
    """

    def __init__(self, **kwargs):
        super().__init__()
        self.param_vector = torch.nn.Parameter(
            torch.tensor(
                [
                    kwargs.get("a1", 0.49484001),
                    kwargs.get("s8", 0.78981345),
                    kwargs.get("a2", 5.73083694),
                ],
                dtype=torch.float64,
                device=torch.device("cpu"),
            )
        )
        self.param = {
            "a1": self.param_vector[0],
            "s8": self.param_vector[1],
            "a2": self.param_vector[2],
        }

    def forward(self, numbers, positions):
        energy = torch.sum(d3.dftd3(numbers, positions, self.param), dim=-1)
        return energy


model = Model()
energy = model.forward(numbers, positions)
print(f"Origin energy: {energy}")
torch.set_printoptions(precision=10)
loss_function = torch.nn.MSELoss(reduction="mean")

target = torch.tensor(
    [-0.001, -0.001],
    dtype=torch.float64,
    device=torch.device("cpu"),
)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.01,
    weight_decay=1e-5,
)

for epoch in range(10000):
    optimizer.zero_grad()
    energy = model(numbers, positions)
    loss = loss_function(energy, target)
    loss_record = torch.sum(torch.abs(energy - target))
    # clip the loss to avoid exploding gradients
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Updated energy: {energy}, loss: {loss_record.item()}")

Origin energy: tensor([-0.0014092580, -0.0057840118], dtype=torch.float64,
       grad_fn=<SumBackward1>)
Updated energy: tensor([-0.0014092580, -0.0057840118], dtype=torch.float64,
       grad_fn=<SumBackward1>), loss: 0.005193269840111606
Updated energy: tensor([-0.0003227659, -0.0015035409], dtype=torch.float64,
       grad_fn=<SumBackward1>), loss: 0.0011807750084821315
Updated energy: tensor([-0.0002718026, -0.0012771653], dtype=torch.float64,
       grad_fn=<SumBackward1>), loss: 0.001005362754098424
Updated energy: tensor([-0.0002552354, -0.0012027120], dtype=torch.float64,
       grad_fn=<SumBackward1>), loss: 0.0009474766551318625
Updated energy: tensor([-0.0002497408, -0.0011776243], dtype=torch.float64,
       grad_fn=<SumBackward1>), loss: 0.0009278835793828931
Updated energy: tensor([-0.0002482154, -0.0011702394], dtype=torch.float64,
       grad_fn=<SumBackward1>), loss: 0.0009220239857470345
Updated energy: tensor([-0.0002479719, -0.0011685226], dtype=torch.float64,
    

In [14]:
torch.sum(energy)

tensor(-0.0037074378, dtype=torch.float64, grad_fn=<SumBackward0>)